# 2. ノンパラメトリック回帰によるATE推定

この notebook では、前の notebook の線形回帰から一歩進んで、**より柔軟な回帰関数を使って平均処置効果（ATE）を推定する**流れを確認します。

スライドとの対応でいうと、

- 回帰関数 $\gamma(d, z) = \mathbb{E}[Y \mid D=d, Z=z]$ を推定する
- その差 $\gamma(1, z) - \gamma(0, z)$ を平均して ATE を求める
- 回帰関数や重み付けの推定を `genriesz` に任せる

という実装になっています。

ここでは標準的な ATE 問題なので、`genriesz` の built-in 関数 `grr_ate` を使います。  
ただし、内部では回帰調整・重み付け・補正付き推定が実行されており、`cross_fit=True` を指定すると交差適合つきの推定、つまり DML 的な処理が自動で行われます。

まずは必要なライブラリを読み込みます。


In [2]:
import numpy as np

from genriesz import (
    grr_ate,
    SquaredGenerator,
    UKLGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
    RBFRandomFourierBasis,
    KNNCatchmentBasis,
)

rng = np.random.default_rng(0)

## データの生成

ここでは、線形回帰では表しにくい**非線形な回帰関数**をもつ人工データを作ります。観測モデルは

$$
Y = f(D, Z) + \varepsilon
$$

です。

- $D \in \{0,1\}$ は処置
- $Z$ は共変量
- $f$ は未知の非線形関数
- $\varepsilon$ はノイズ

処置割り当ては $Z$ に依存しているので、ここでも交絡があります。したがって、単純に処置群平均と対照群平均の差を取るだけでは ATE は得られません。

今回の潜在アウトカムは、$Z$ の二次項や交互作用、$\sin$ を含むように作ってあります。  
そのため、「真の関数形を完全には知らない」という現実に近い状況で、どの基底や近似方法がうまく働くかを観察できます。

さらに、この notebook では潜在アウトカムの平均差が**標本レベルでちょうど 3**になるように正規化しています。  
したがって、推定結果は「3 にどれだけ近いか」で比較できます。


In [5]:
# Data-generating process
## ATEの計算のために（分析では使わない）大量のデータを用意します
n_sim = 100000
## 実際に分析に使うデータを用意します．
n = 1000
d_z = 5

Z = rng.normal(size=(n, d_z))

# Treatment assignment: e(Z) = sigmoid(a'Z)
logits = 0.7 * Z[:, 0] - 0.3 * Z[:, 1]
e = 1.0 / (1.0 + np.exp(-logits))
D = rng.binomial(1, e, size=n).astype(float)

# Potential outcomes (constant effect for simplicity)
tau = 3.0
beta0 = np.random.uniform(0, 1, 4)
mu0 = beta0[0] * Z[:, 0] + beta0[1] * Z[:, 1] ** 2 + beta0[2] * Z[:, 0] * Z[:, 1] + beta0[3] * np.sin(Z[:, 2])

beta1 = np.random.uniform(0, 1, 2)
mu1 = beta1[0] * Z[:, 0] + beta1[1] * Z[:, 1] ** 2

mu0 = mu0 / np.mean(mu0)
mu1 = mu1 / np.mean(mu1) * (1 + tau)

Y0 = mu0 + rng.normal(scale=1.0, size=n)
Y1 = mu1 + rng.normal(scale=1.0, size=n)

Y = D * Y1 + (1.0 - D) * Y0

# Regressor matrix X = [D, Z...]
X = np.column_stack([D, Z])

Y = Y[:n]
X = X[:n]
print("X shape:", X.shape, "Y shape:", Y.shape)

X shape: (1000, 6) Y shape: (1000,)


## 多項式基底による推定

最初に、回帰関数と Riesz 表現量の近似に**二次の多項式基底**を使います。

実装上のポイントは 2 つあります。

1. `PolynomialBasis(degree=2)` で、共変量 $Z$ の一次・二次の特徴量を作る
2. `TreatmentInteractionBasis` で、その基底を処置 $D$ と相互作用させる

こうすることで、処置群と対照群で異なる回帰曲面を表現しやすくなります。ATE のような二値処置の問題では自然な設計です。

`grr_ate` は複数の推定量をまとめて返します。

- `ra`: 回帰調整型
- `rw`: 重み付け型
- `arw`: 補正付き重み付け型
- `tmle`: 標的化更新を入れた型

また、`cross_fit=True` としているので、局外母数の推定は out-of-fold で行われます。  
つまり、**標準的な ATE 問題に対する DML 的な実装**になっています。


In [4]:
# Basis on Z, then interact with D (ATE-friendly)
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

# Generator: Squared loss (always safe / no domain constraints)
gen = SquaredGenerator(C=0.0).as_generator()

res_poly = grr_ate(
    X=X,
    Y=Y,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_poly.summary_text())


ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.7153645757945919, max_abs_smd_weighted=0.02558126061975452, ess_treated=402.6245935618626, ess_control=383.6878723951534

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 2.97912      0.167585         [ 2.65066,  3.30758]           0
RW                 3.01777      0.408751         [ 2.21663,  3.81891]    1.55e-13
ARW                2.97912      0.182363         [ 2.62169,  3.33654]           0
TMLE               2.97912      0.182363         [ 2.62169,  3.33654]           0


多項式基底の結果では、推定値が真値 3 に比較的近くなっているはずです。

これは今回の DGP に

- 二次項
- 交互作用
- 比較的なめらかな非線形性

が含まれており、二次多項式がかなりうまく近似できるからです。

ここで大切なのは、**ATE 推定の成否は回帰関数や重みの近似精度に依存する**という点です。  
十分に表現力のある基底を使えば、線形モデルよりも柔軟に因果効果を推定できます。


## RBF ランダム特徴による推定

次に、再生核ヒルベルト空間を近似するために、**RBF ランダムフーリエ特徴**を使います。

RBF 基底は非常に柔軟で、なめらかな非線形関数を広く近似できます。  
一方で、特徴量数・帯域幅・正則化の設定に結果がかなり左右されます。

このセルでは、

- `RBFRandomFourierBasis` でランダム特徴を生成し
- それを `TreatmentInteractionBasis` で処置と相互作用させ
- 同じ `grr_ate` を適用

しています。

つまり、**推定アルゴリズムの枠組みは同じまま、関数近似の仕方だけを変えている**と見ることができます。


In [40]:
psi_rff = RBFRandomFourierBasis(
    n_features=500,
    sigma=1.0,
    standardize=True,
    random_state=0,
)
phi_rff = TreatmentInteractionBasis(base_basis=psi_rff)

res_rff = grr_ate(
    X=X,
    Y=Y,
    basis=phi_rff,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rff.summary_text())


ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.6662380087434316, max_abs_smd_weighted=0.2443503805709245, ess_treated=445.4983650116201, ess_control=432.6136266240531

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                  4.4961      0.265099         [ 3.97651,  5.01568]           0
RW                 5.23514      0.804863         [ 3.65764,  6.81264]     7.8e-11
ARW                4.10464      0.568956         [ 2.98951,  5.21978]    5.42e-13
TMLE                4.1691      0.567248         [ 3.05732,  5.28089]    1.99e-13


RBF ランダム特徴の結果が多項式基底より不安定になることがあります。

これは RBF が悪いというより、

- 今回の標本サイズ
- ランダム特徴の数
- 帯域幅 $\sigma$
- 正則化の強さ

の組み合わせが、今回の DGP にとって最適とは限らないからです。

この比較から分かるのは、**柔軟なモデルを使えば自動的に良くなるわけではない**ということです。  
表現力が高いぶん、チューニングや標本サイズへの依存も大きくなります。


## ニューラルネットワーク特徴による推定

最後に、ニューラルネットワークから得られる埋め込み特徴を使います。

ここでは `MLPEmbeddingNet` で特徴抽出器を作り、`TorchEmbeddingBasis` に渡しています。  
ただし、この notebook では計算を軽く保つために**埋め込みネットワーク自体は事前学習していません**。つまり、学習済み表現ではなく、ランダム初期化に近い特徴をそのまま使っています。

したがって、このセルの目的は「最強のニューラルネットを作ること」ではなく、

- `genriesz` がニューラルネット由来の特徴量も扱えること
- 基底の選び方次第で推定結果が変わること

を確認することにあります。

より実務的には、別タスクで学習した表現を凍結して使う、あるいは特徴抽出器も含めて丁寧に調整するほうが自然です。


In [50]:
import torch
from genriesz.torch_basis import MLPEmbeddingNet, TorchEmbeddingBasis

torch.manual_seed(0)

net = MLPEmbeddingNet(input_dim=X.shape[1], hidden_dims=(64,), output_dim=32)
# (Optional) Train net here on a separate task.
# For a lightweight demo, we skip training and just use the random initialization.
nn_basis = TorchEmbeddingBasis(net, include_bias=True, device="cpu")

res_nn = grr_ate(
    X=X,
    Y=Y,
    basis=nn_basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_nn.summary_text())

ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.6662380087434316, max_abs_smd_weighted=0.11310789974578712, ess_treated=408.804419100671, ess_control=416.27665102482666

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 2.20889        0.1575          [ 1.9002,  2.51758]           0
RW                 2.25174      0.571085         [ 1.13243,  3.37104]    8.05e-05
ARW                2.36108      0.383275         [ 1.60987,  3.11229]    7.26e-10
TMLE               2.36555      0.382785          [ 1.6153,  3.11579]    6.42e-10


ニューラルネットワーク特徴の結果は、今回の設定では「ある程度は当たるが、多項式基底ほど安定ではない」という挙動になっても不思議ではありません。

理由は、先ほど述べた通り、この notebook では埋め込みを十分に学習していないからです。  
それでも、

- 非線形特徴を使った ATE 推定の流れ
- `genriesz` で複数の推定量を同時に比較できること
- `cross_fit=True` によって DML 的な処理が自動で入ること

は確認できます。

この notebook の焦点は、「どの基底が絶対に最良か」を決めることではなく、**回帰関数の近似方法の違いが ATE 推定にどう影響するか**を見ることです。


## この notebook のまとめ

ここでは、ATE 推定を線形回帰からノンパラメトリック回帰へ拡張しました。

- ATE は回帰関数の差 $\gamma(1,z)-\gamma(0,z)$ の平均として書ける。
- したがって、回帰関数をうまく近似できれば ATE も推定できる。
- `genriesz` の `grr_ate` を使うと、RA・RW・ARW・TMLE をまとめて実行できる。
- `cross_fit=True` を指定すると、標準的な ATE 問題に対して DML 的な交差適合が自動で行われる。
- ただし、推定精度は基底や特徴量設計に大きく依存する。

次の notebook では、標準的な ATE の built-in 関数から少し離れて、**差分の差法や平均限界効果を「線型汎函数 $m$ を自分で書く」一般形として実装する**方向へ進みます。
